# Máquinas de vetores de suporte (SVM)

**Objetivo:** ver a margem e os vetores de suporte de uma SVM linear e, no caso clássico dos dados 'em círculos', ver o **truque do kernel** (RBF) resolver o que nenhuma reta consegue.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Dados não separáveis por uma reta

O `make_circles` gera um anel de uma classe em volta de um núcleo da outra: não há reta que os separe.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.svm import SVC

X, y = make_circles(n_samples=300, factor=0.4, noise=0.12, random_state=SEMENTE)
print("X:", X.shape, "| classes:", np.unique(y))

figura = go.Figure(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                              marker=dict(color=y, colorscale="Bluered", size=6)))
figura.update_layout(title="Dados em circulos: nenhuma reta separa",
                     height=380, margin=dict(l=10, r=10, t=50, b=10), showlegend=False)
figura.show()

## 2. SVM linear × SVM com kernel RBF

Treinamos as duas e comparamos a acurácia. A linear fracassa; a RBF, que projeta os dados implicitamente para onde eles ficam separáveis, acerta.

In [ ]:
svm_linear = SVC(kernel="linear").fit(X, y)
svm_rbf = SVC(kernel="rbf", C=1.0, gamma=1.0).fit(X, y)
print("acuracia SVM linear:", round(svm_linear.score(X, y), 3))
print("acuracia SVM RBF:   ", round(svm_rbf.score(X, y), 3))
print("vetores de suporte da RBF:", svm_rbf.support_vectors_.shape[0], "de", len(X))

## 3. As fronteiras lado a lado

Pintamos a região prevista por cada modelo. A da SVM linear é um semiplano (inútil aqui); a da RBF é um anel que envolve o núcleo. Os pontos maiores são os **vetores de suporte** — os únicos que definem a fronteira.

In [ ]:
from plotly.subplots import make_subplots
passo = 0.03
gx, gy = np.meshgrid(np.arange(X[:, 0].min()-0.3, X[:, 0].max()+0.3, passo),
                     np.arange(X[:, 1].min()-0.3, X[:, 1].max()+0.3, passo))
grade = np.c_[gx.ravel(), gy.ravel()]

figura = make_subplots(rows=1, cols=2, subplot_titles=("SVM linear", "SVM RBF"))
coluna = 1
for modelo in [svm_linear, svm_rbf]:
    zz = modelo.predict(grade).reshape(gx.shape)
    figura.add_trace(go.Heatmap(x=gx[0], y=gy[:, 0], z=zz, showscale=False,
                                colorscale="Bluered", opacity=0.3), row=1, col=coluna)
    figura.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                                marker=dict(color=y, colorscale="Bluered", size=5),
                                showlegend=False), row=1, col=coluna)
    sv = modelo.support_vectors_
    figura.add_trace(go.Scatter(x=sv[:, 0], y=sv[:, 1], mode="markers",
                                marker=dict(color="rgba(0,0,0,0)", size=11,
                                            line=dict(width=1.5, color=VERDE)),
                                showlegend=False), row=1, col=coluna)
    coluna += 1
figura.update_layout(title="Fronteiras: linear falha, RBF resolve (vetores de suporte em verde)",
                     height=400, margin=dict(l=10, r=10, t=60, b=10))
figura.show()

## 4. O efeito de C e γ

Varremos alguns valores de $\gamma$ (mantendo $C$) e medimos a acurácia de validação cruzada. $\gamma$ grande demais memoriza (overfitting); pequeno demais suaviza a ponto de perder o anel.

In [ ]:
from sklearn.model_selection import cross_val_score

for gamma in [0.1, 1.0, 10.0, 100.0]:
    modelo = SVC(kernel="rbf", C=1.0, gamma=gamma)
    ac = cross_val_score(modelo, X, y, cv=5).mean()
    ac_treino = modelo.fit(X, y).score(X, y)
    print("gamma =", str(gamma).rjust(5),
          "| treino", round(ac_treino, 3), "| validacao CV", round(ac, 3))

## Exercício

Na varredura acima, para qual $\gamma$ a acurácia de treino é altíssima mas a de validação cai? Como esse padrão se chama e como corrigi-lo?

<details><summary>Ver resposta</summary>

Para o $\gamma$ **mais alto** (100): o treino fica quase perfeito enquanto a validação cruzada cai. É **overfitting** — cada ponto vira uma bolha da sua classe, memorizando o treino sem capturar o anel real. Corrige-se **reduzindo $\gamma$** (e escolhendo $C$ e $\gamma$ por validação cruzada, por exemplo com `GridSearchCV`).

</details>